# Recreate Figure 1: SQNR(bits) vs Bitrate on Gaussian Source

This notebook recreates the paper-style Gaussian-source evaluation.

It follows the metric in the paper:

- draw i.i.d. blocks `w ~ N(0, 1)`,
- quantize each block,
- compute empirical `MSE = mean(||w - q(w)||^2 / D)`,
- report `SQNR_bits = -0.5 * log2(MSE)`,
- compare against the Shannon line `SQNR_bits = R`.

Curves included here:

- `LLVQ/Leech (spherical shaping)` using this repo's generated Leech quantizer,
- `QuIP# official E8P/RVQ` using upstream `Cornell-RelaxML/quip-sharp` codebook classes from `third_party/quip-sharp`,
- `E8 (cubic)` using `impl.gaussian_baseline_quantizers.CubicE8Quantizer`,
- `Uniform` using `impl.gaussian_baseline_quantizers.UniformScalarQuantizer`.

This does not run the full QuIP# LLM pipeline: no Hessian LDLQ, randomized Hadamards, CUDA inference kernels, or fine-tuning are used for this Gaussian-source codebook plot.


In [ ]:
from __future__ import annotations

import math
import os
import sys
import time
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "impl").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from impl.gaussian_baseline_quantizers import (
    CubicE8Quantizer,
    QuipSharpE8Quantizer,
    UniformScalarQuantizer,
    mse_to_sqnr_bits,
)
from impl.leech_lattice_vector_quantizer import DIM, LeechLatticeVectorQuantizer, squared_distance

print(ROOT)


## Configuration

The structured Leech nearest-neighbor search avoids materializing the codebook, but it is still pure Python. Start with a small sample count, then increase `N_SAMPLES` for smoother curves.

The shell cutoff `m=51` reaches exactly `3.0` bits/dim for the cumulative Leech ball by the theta-series count. It can be slow; remove it while iterating.

The QuIP# official points use `E8P12` at 2 bits/dim and `E8P12RVQ3B` at 3 bits/dim.


In [ ]:
SEED = 0
N_SAMPLES = 8

# Cumulative Leech ball cutoffs. m=51 reaches 3.0 bits/dim.
SHELL_CUTOFFS = [2, 3, 4, 5, 6, 8, 10, 13, 20, 30, 40, 51]

# Baseline rates. Integer bits keep the simple baselines honest and fast.
BASELINE_RATES = [1.0, 2.0, 3.0]

# Spherical shaping needs a global scale: beta * Q(w / beta).
# Coarse by default because each beta invokes the Leech search.
LLVQ_SCALE_GRID = np.linspace(0.35, 1.25, 10)

rng = np.random.default_rng(SEED)
samples = rng.normal(0.0, 1.0, size=(N_SAMPLES, DIM))
samples.shape


## Evaluation Helpers

In [ ]:
def evaluate_spherical_shaping(
    max_shell: int,
    samples: np.ndarray,
    scale_grid: np.ndarray = LLVQ_SCALE_GRID,
) -> dict:
    """Evaluate LLVQ spherical shaping with optimized global scale beta."""
    q = LeechLatticeVectorQuantizer(max_shell=max_shell)
    start = time.perf_counter()
    best = None

    for beta in scale_grid:
        sqerr = 0.0
        indices = []
        for w in samples:
            index = q.quantize(w / beta)
            recon = beta * np.array(q.dequantize_lattice(index), dtype=float)
            sqerr += float(np.sum((w - recon) ** 2))
            indices.append(index)
        mse = sqerr / (samples.shape[0] * DIM)
        if best is None or mse < best["mse"]:
            best = {"mse": mse, "scale": float(beta), "first_index": indices[0]}

    assert best is not None
    rate = q.shape_bits / DIM
    sqnr = mse_to_sqnr_bits(best["mse"])
    return {
        "method": "LLVQ/Leech (spherical shaping)",
        "max_shell": max_shell,
        "rate": rate,
        "mse": best["mse"],
        "sqnr_bits": sqnr,
        "retention_pct": 100.0 * sqnr / rate,
        "total_count": q.total_count,
        "shape_bits": q.shape_bits,
        "scale": best["scale"],
        "seconds": time.perf_counter() - start,
        "first_index": best["first_index"],
    }

def evaluate_uniform(samples: np.ndarray, rates: list[float]) -> list[dict]:
    rows = []
    for rate in rates:
        if abs(rate - round(rate)) > 1e-9:
            continue
        result = UniformScalarQuantizer(bits=int(round(rate))).evaluate(samples)
        rows.append({
            "method": result.method,
            "rate": result.rate,
            "mse": result.mse,
            "sqnr_bits": result.sqnr_bits,
            "scale": result.scale,
            **result.metadata,
        })
    return rows


def evaluate_e8_cubic(samples: np.ndarray, radii: list[float] = [0.5, 1.0, 1.5, 2.0]) -> list[dict]:
    rows = []
    for radius in radii:
        result = CubicE8Quantizer(radius=radius).evaluate(samples)
        rows.append({
            "method": result.method,
            "rate": result.rate,
            "mse": result.mse,
            "sqnr_bits": result.sqnr_bits,
            "scale": result.scale,
            **result.metadata,
        })
    return rows


def evaluate_quip_official(samples: np.ndarray, names: list[str] = ["E8P12", "E8P12RVQ3B"]) -> list[dict]:
    rows = []
    for name in names:
        result = QuipSharpE8Quantizer(name, repo_root=ROOT).evaluate(samples)
        rows.append({
            "method": result.method,
            "rate": result.rate,
            "mse": result.mse,
            "sqnr_bits": result.sqnr_bits,
            "scale": result.scale,
            **result.metadata,
        })
    return rows


def print_row(row: dict) -> None:
    maybe_shell = f"m<={row['max_shell']:2d}  " if "max_shell" in row else "      "
    print(
        f"{row['method']:<34} {maybe_shell}"
        f"R={row['rate']:.4f}  "
        f"MSE={row['mse']:.6f}  "
        f"SQNRbits={row['sqnr_bits']:.4f}  "
        f"scale={row.get('scale', float('nan')):.3f}"
    )


## Run the Gaussian Benchmark

This cell is the expensive part. With the default small sample count it is meant to be interactive rather than publication-grade.

For fast iteration, temporarily set `SHELL_CUTOFFS = [2, 3, 4, 5, 10]`. For a full 0--3 bits/dim curve, keep `51` in the list.

For LLVQ spherical shaping, the notebook optimizes a global scale `beta` and reconstructs `beta * Q(w / beta)`. This is essential: increasing the ball radius without reducing the scale only adds farther lattice points and can plateau once the sample norms are covered.


In [ ]:
llvq_results = []
for max_shell in SHELL_CUTOFFS:
    row = evaluate_spherical_shaping(max_shell, samples)
    llvq_results.append(row)
    print_row(row)

uniform_results = evaluate_uniform(samples, BASELINE_RATES)
e8_cubic_results = evaluate_e8_cubic(samples)
quip_results = evaluate_quip_official(samples)

for row in uniform_results + e8_cubic_results + quip_results:
    print_row(row)


## Plot Figure 1 Style Curve

`SQNR_bits = R` is the Shannon limit for a unit-variance Gaussian source under the paper's convention.

In [ ]:
all_results = llvq_results + uniform_results + e8_cubic_results + quip_results

fig, ax = plt.subplots(figsize=(8.0, 5.2), dpi=140)
x = np.linspace(0.0, 3.05, 200)
ax.plot(x, x, color="black", linewidth=1.5, linestyle="--", label="Shannon bound")

styles = {
    "LLVQ/Leech (spherical shaping)": dict(marker="o", linewidth=2.2),
    "QuIP# official E8P/RVQ": dict(marker="s", linewidth=1.8),
    "E8 (cubic)": dict(marker="^", linewidth=1.8),
    "Uniform": dict(marker="d", linewidth=1.8),
}

for method, style in styles.items():
    rows = sorted([r for r in all_results if r["method"] == method], key=lambda r: r["rate"])
    if not rows:
        continue
    rates = np.array([r["rate"] for r in rows])
    sqnr = np.array([r["sqnr_bits"] for r in rows])
    ax.plot(rates, sqnr, label=method, **style)

# Paper Table 7 sanity marker for spherical shaping at R=2, MSE=0.084.
paper_r = 2.0
paper_mse = 0.084
paper_sqnr = mse_to_sqnr_bits(paper_mse)
ax.scatter([paper_r], [paper_sqnr], marker="x", s=80, color="tab:red", label="Paper Table 7 spherical point")

ax.set_title("SQNRbits vs bitrate on Gaussian source")
ax.set_xlabel("Bitrate R (bits/dim)")
ax.set_ylabel("SQNRbits")
ax.set_xlim(0.0, 3.05)
ax.set_ylim(0.0, 3.05)
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right", fontsize=8)
plt.show()


## Inspect One Quantized Block

The index decodes to an integer representative in `Lint`; the actual Leech lattice reconstruction is scaled by `1/sqrt(8)`. This is why the dequantized values are real numbers even though the index hierarchy is integer-valued.

In [ ]:
q = LeechLatticeVectorQuantizer(max_shell=SHELL_CUTOFFS[-1])
w = samples[0]
idx = q.quantize(w)
ranked = q.unrank(idx)

print("input:", np.round(w, 4))
print("index:", idx)
print("address:", ranked)
print("integer representative:", q.dequantize(idx))
print("scaled reconstruction:", np.round(q.dequantize_lattice(idx), 4))
print("block MSE:", squared_distance(w, q.dequantize_lattice(idx)) / DIM)
